## IMPORTATION DES BIBLIOTHEQUES ET PARAMETRES GLOBAUX

In [ ]:
# 1. Imports
import os
import time
import math
import importlib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from pyproj import CRS
import csv
from pathlib import Path; import ipynbname; 
import sys

pd.set_option("display.max_columns", None)
# 2. Constantes et variables globales
%matplotlib qt
base_path = ipynbname.path().parent.parent.parent
path_data = os.path.join(base_path,'DATA')
# Ajouter le dossier scripts au path
scripts_path = base_path/ "Code"  / "scripts"
sys.path.append(str(scripts_path.resolve()))


# Recharger autoreload pour prendre en compte les dernières modifs
%reload_ext autoreload

# 2️⃣ Recharger automatiquement tous les modules à chaque exécution
%autoreload 2

# Charger les modules
from fonctions_annexes_biodiv import *
from formatage_biodiv import *
from formatage_geo import *
from formatage_fusion import *
from formatage_traitement import *

In [ ]:
base_path = ipynbname.path().parent.parent.parent
base_path

## Apercu

In [ ]:
# Aperçu des données SIG
# -------------------------
# Paramètres
PATH_SIG= base_path  / "SIG" / "SIG_Global"
WORLD_TERRESTRE_FILE = PATH_SIG / "geoBoundariesCGAZ_ADM0.shp"
WORLD_MARITIME_FILE = PATH_SIG / "eez_v11.gpkg"

# Charger les données SIG
world_terrestre = gpd.read_file(WORLD_TERRESTRE_FILE)
world_maritime = gpd.read_file(WORLD_MARITIME_FILE)

# Affichage rapide
print("✅ Données terrestres chargées :")
world_terrestre.head()

In [ ]:
# Aperçu des données de biodiversité
# -------------------------
# Chemin vers le fichier GBIF
country_name = "Gabon"
filename = f"extractGBIF_{country_name}.csv"
filepath = base_path / "Data" / "GBIF" / "raw" / f"GBIF_{country_name}" / f"extractGBIF_{country_name}.csv"

# Aperçu complet
df_apercu = apercu_biodiv(filepath,colonnes_a_importer=None, n_lignes=5)

In [ ]:
# Exemple d'utilisation avec df_apercu
apercu_missing(df_apercu)

In [ ]:
# Exemple d'utilisation avec df_apercu
col='basisOfRecord'
apercu_occurrence_status(df_apercu,col)


## Traitement

In [ ]:
# Paramètres
country_name = "Gabon"
grid_size_km = 100  # taille de la maille
bornes_temporelles = [1900,1990,2000,2010,2020,2025]  
path_sig= base_path  / "SIG" / "SIG_Global"
path_data = base_path / "Data"
cle_geo = f"codeMaille{grid_size_km}Km"
cle_ID = "speciesKey"
source = "GBIF"

# Appel de la fonction
df_final_gabon = traiter_pays_et_maille(
    country_name=country_name,
    grid_size_km=grid_size_km,
    bornes_temporelles=bornes_temporelles,
    path_data=path_data,
    path_sig=path_sig,
    cle_geo=cle_geo,
    cle_ID=cle_ID,
    source=source,
    fusion=True,  # si tu veux appliquer la fusion des mailles
    var_obs="nombreObs",
    seuil_fusion=1000,
    methode_fusion="barycentre",
    max_distance=100
)

# Vérification rapide
df_final_gabon.head()


In [ ]:
path_sig

## OLD

In [ ]:
# ===============================
# 1️⃣ Définir le pays et les paramètres
# ===============================
country_name = "Gabon"        # exemple
grid_size_km = 10              # taille de maille
cle_geo = f"codeMaille{grid_size_km}Km"
cle_ID = "speciesKey"          # clé des espèces
source = "GBIF"

# ===============================
# 2️⃣ Charger les données géographiques
# ===============================
world_terrestre, world_maritime = charger_donnees_geo(path_data)

# Vérifier les colonnes
print(world_terrestre.columns)
print(world_maritime.columns)

# Filtrer pour le pays
country_terrestre = world_terrestre[world_terrestre['shapeName'] == country_name]
country_maritime = world_maritime[world_maritime['ISO_TER1'] == country_terrestre['shapeGroup'].iloc[0]]  # correspondance code

print(f"Nombre de polygones terrestres : {len(country_terrestre)}")
print(f"Nombre de polygones maritimes : {len(country_maritime)}")

# Visualisation rapide
fig, ax = plt.subplots(figsize=(8,8))
world_terrestre.plot(ax=ax, color='lightgrey', edgecolor='black')
country_terrestre.plot(ax=ax, color='green', edgecolor='red')
if not country_maritime.empty:
    country_maritime.plot(ax=ax, color='blue', alpha=0.3)
plt.title(f"{country_name} - Terrestre et Maritime")
plt.show()

# ===============================
# 3️⃣ Créer la grille pour le pays
# ===============================
country_grid_terrestre, country_grid_maritime, country_grid_combined = creer_grille_pays(
    path_data, country_name, grid_size_km,
    world_terrestre, world_maritime, cle_geo, source=source
)

# Affichage rapide de la grille
fig, ax = plt.subplots(figsize=(8,8))
country_grid_combined.plot(ax=ax, color='lightblue', edgecolor='grey', alpha=0.5)
country_terrestre.plot(ax=ax, edgecolor='red', linewidth=2, facecolor='none')
plt.title(f"Grille combinée pour {country_name}")
plt.show()

# ===============================
# 4️⃣ Explorer les données brutes GBIF
# ===============================
path_fichier = os.path.join(path_data, source,"raw", f"{source}_{country_name}", f"extract{source}_{country_name}.csv")

# Détection du séparateur
sep = detect_sep(path_fichier)

# Lire juste un petit échantillon
df_sample = pd.read_csv(path_fichier, sep=sep, nrows=1000)
print(df_sample.head())
print(f"Colonnes disponibles : {df_sample.columns.tolist()}")


In [ ]:

filtered_countries=["Gabon"]

tailles_maille = [10]
cle_ID="speciesKey"
source="GBIF"
n_donnees_par_maille=500
fusion=True

# Bornes temporelles modifiables
bornes_temporelles = [1800, 1990,2010, 2024]  # Vous pouvez les ajuster ici
chaine_bornes = "_".join(map(str, bornes_temporelles))

for country_name in filtered_countries:
    country_name_underscore=country_name.replace('_', ' ')
  
    sig_path = os.path.join(path,'SIG_global')
    path_data = os.path.join(path,'DATA')
    path_fichier = os.path.join(path_data,source,"raw",f"{source}_{country_name}", f"extract{source}_{country_name}.csv")  
    world_terrestre=gpd.read_file(os.path.join(sig_path, "geoBoundariesCGAZ_ADM0.shp"))

    #largeur_recommandee=calcul_largeur_maille_pays(country_name, world_terrestre, path_fichier, 
    
    
    for grid_size_km in tailles_maille:
        cle_geo = f"codeMaille{grid_size_km}Km"
        print(f"\n📢 Traitement pour le pays : {country_name} avec une taille de maille : {grid_size_km} km\n")
        traiter_pays_et_maille(country_name, grid_size_km, bornes_temporelles, path, cle_geo, cle_ID,source=source,
        fusion=fusion,var_obs='nombreObs', seuil_fusion=n_donnees_par_maille, methode_fusion="barycentre", max_distance=100)
    #     rename_sig_files(country_name, largeur_recommandee,path_data,source=source)



### Fusion

In [ ]:
country_name="France"
grid_size_km = 2
ecosysteme='terrestre'
bornes_temporelles = [1800, 1990,2010, 2024] 
cle_ID="speciesKey"
source="GBIF"
n_donnees_par_maille=5_000
fusion=True
cle_geo = f"codeMaille{grid_size_km}Km"
path_data = os.path.join(path,'DATA')


fusionner_depuis_fichiers(
    country_name, grid_size_km, ecosysteme, bornes_temporelles,
    path_data, cle_geo, cle_ID,
    source="GBIF", var_obs="nombreObs",
    seuil_fusion=n_donnees_par_maille, methode_fusion="barycentre", max_distance=100)

## INAT

In [ ]:
import pandas as pd
import os
from datetime import datetime
import pycountry

file_path = r"C:\Users\Aubin\Documents\MANTIS\Data\iNat\iNat_full.csv"

# Détecter le séparateur
with open(file_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
sep = "\t" if "\t" in first_line else ","
print(f"Séparateur détecté : {sep}")

# Colonnes à garder
colonnes_a_garder = ['class', 'decimalLongitude', 'decimalLatitude', 'eventDate', 'year', 'order', 'family', 'species', 'genus', 'occurrenceStatus', 'taxonRank', 'speciesKey', 'occurrenceID', 'countryCode', 'phylum', 'verbatimScientificName', 'kingdom', 'individualCount']
# Dictionnaire ISO2 -> nom pays

today = datetime.today().strftime("%Y%m%d")
base_path = r"C:\Users\Aubin\Documents\MANTIS\DATA\iNat"
chunksize = 1_000_000

# dictionnaire alpha2 -> alpha3
code2to3 = {c.alpha_2: c.alpha_3 for c in pycountry.countries}

# lecture d’un chunk
for chunk_num, chunk in enumerate(pd.read_csv(file_path, sep=sep, chunksize=chunksize, on_bad_lines="skip", encoding="utf-8"), start=1):
    print(f"chunk n°{chunk_num}")
    
    df_reduit = chunk[colonnes_a_garder].copy()
    
    # convertir ISO2 -> ISO3
    df_reduit['countryCode3'] = df_reduit['countryCode'].map(code2to3)
    
    # merge avec shapefile
    df_join = df_reduit.merge(
        world_terrestre[['shapeGroup', 'shapeName']], 
        left_on='countryCode3', 
        right_on='shapeGroup', 
        how='inner'
    )

    for pays, data_pays in df_join.groupby("shapeName"):
        dossier = os.path.join(base_path, "raw",f"iNat_{pays}" )
        os.makedirs(dossier, exist_ok=True)
        
        
        # ajouter le numéro du chunk dans le nom du fichier
        fichier_csv = os.path.join(dossier, f"extractiNat_{pays}_{today}_chunk{chunk_num}.csv")
        data_pays = data_pays.drop(columns=[col for col in ["nomPays", "shapeName"] if col in data_pays.columns])
        
        # écriture CSV
        data_pays.to_csv(fichier_csv, index=False)
        #print(f"Sauvegardé : {fichier_csv}")


In [ ]:
import pandas as pd
import os
import glob

# Saisie manuelle de la date au format YYYYMMDD
today = "20250914"

base_path = r"C:\Users\Aubin\Documents\MANTIS\DATA\iNat"

# Boucler sur tous les pays connus
for pays in world_terrestre['shapeName']:
    
    dossier = os.path.join(base_path, f"iNat_{pays}", "raw")
    
    # Chercher tous les fichiers chunk de ce pays
    fichiers_temp = glob.glob(os.path.join(dossier, f"extractiNat_{pays}_{today}_chunk*.csv"))
    print(len(fichiers_temp))
    if not fichiers_temp:
        continue
    
    # Lire tous les fichiers temporaires et concaténer
    df_list = [pd.read_csv(f) for f in fichiers_temp]
    df_final = pd.concat(df_list, ignore_index=True)
    
    # Sauvegarder le fichier final
    fichier_final_csv = os.path.join(dossier, f"extractiNat_{pays}_{today}.csv")
    df_final.to_csv(fichier_final_csv, index=False)
    print(f"Fichier final CSV créé : {fichier_final_csv}")
    
    # Supprimer les fichiers temporaires
    for f in fichiers_temp:
        os.remove(f)


In [ ]:
import os
import glob

racine = r"C:\Users\Aubin\Documents\MANTIS\DATA\iNat"

# motif qui parcourt tous les pays -> raw -> sous-dossiers éventuels -> fichiers ciblés
pattern = os.path.join(racine, "*", "raw", "**", "extractiNat_*_20250913.csv")

# recherche récursive
for fichier in glob.glob(pattern, recursive=True):
    try:
        os.remove(fichier)
        print(f"Supprimé : {fichier}")
    except Exception as e:
        print(f"Erreur sur {fichier} : {e}")


## SCRIPT ENTIER

In [ ]:

filtered_countries=["Gabon"]

tailles_maille = [10]
cle_ID="speciesKey"
source="GBIF"
n_donnees_par_maille=500
fusion=True

# Bornes temporelles modifiables
bornes_temporelles = [1800, 1990,2010, 2024]  # Vous pouvez les ajuster ici
chaine_bornes = "_".join(map(str, bornes_temporelles))

for country_name in filtered_countries:
    country_name_underscore=country_name.replace('_', ' ')
  
    sig_path = os.path.join(path,'SIG_global')
    path_data = os.path.join(path,'DATA')
    path_fichier = os.path.join(path_data,source,"raw",f"{source}_{country_name}", f"extract{source}_{country_name}.csv")  
    world_terrestre=gpd.read_file(os.path.join(sig_path, "geoBoundariesCGAZ_ADM0.shp"))

    #largeur_recommandee=calcul_largeur_maille_pays(country_name, world_terrestre, path_fichier, 
    
    
    for grid_size_km in tailles_maille:
        cle_geo = f"codeMaille{grid_size_km}Km"
        print(f"\n📢 Traitement pour le pays : {country_name} avec une taille de maille : {grid_size_km} km\n")
        traiter_pays_et_maille(country_name, grid_size_km, bornes_temporelles, path, cle_geo, cle_ID,source=source,
        fusion=fusion,var_obs='nombreObs', seuil_fusion=n_donnees_par_maille, methode_fusion="barycentre", max_distance=100)
    #     rename_sig_files(country_name, largeur_recommandee,path_data,source=source)




### Fusion  depuis fichiers

In [ ]:
country_name="France"
grid_size_km = 2
ecosysteme='terrestre'
bornes_temporelles = [1800, 1990,2010, 2024] 
cle_ID="speciesKey"
source="GBIF"
n_donnees_par_maille=5_000
fusion=True
cle_geo = f"codeMaille{grid_size_km}Km"
path_data = os.path.join(path,'DATA')


fusionner_depuis_fichiers(
    country_name, grid_size_km, ecosysteme, bornes_temporelles,
    path_data, cle_geo, cle_ID,
    source="GBIF", var_obs="nombreObs",
    seuil_fusion=n_donnees_par_maille, methode_fusion="barycentre", max_distance=100)